# MNIST MLP3: AdamW versus AdaptiveSpectralGuard — Stabilized V2

This notebook tests a stabilized configuration motivated by the first eight epochs of the original adaptive run.

The first run showed two specific controller problems:

1. the shell-$\beta_E$ channel sat almost exactly on its per-step cap whenever it was active;
2. low, noisy ECS confidence repeatedly turned **all** FC1 correction off even after WeightWatcher reported $\alpha<2$.

V2 changes only the controller and layer policy:

- **three-checkpoint confidence smoothing** using an exponential moving average;
- **separate volume and shape confidence gates**;
- a **volume-confidence floor** when $\alpha\leq2.05$, so uncertain ECS support cannot disable all trace-log branch protection below the boundary;
- a stricter shape gate requiring both a reliable positive $\beta_E$ and $\alpha\leq2.05$;
- a raw-confidence veto for the shape channel on exceptionally unstable checkpoints;
- a shell-$\beta_E$ deadband;
- lower shape scales and caps:
  - FC1: $5\%\rightarrow2\%$ of the AdamW step;
  - FC2: $2\%\rightarrow0.75\%$ of the AdamW step.

The trace-log and shape channels now have distinct gains,

\[
g_{\ell,e}^{(T)}
=
g_{\ell,e}^{\rm base}\,
C_{\ell,e}^{(T)}\,
Q_{\ell,e},
\qquad
g_{\ell,e}^{(\beta)}
=
g_{\ell,e}^{\rm base}\,
C_{\ell,e}^{(\beta)}\,
Q_{\ell,e},
\]

so an unstable spectral-shape estimate can turn off the $\beta_E$ channel without turning off one-sided trace-log protection.


In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import weightwatcher as ww
from IPython.display import display

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "adaptive_spectral_guard").is_dir():
        ROOT = candidate
        break
    nested = candidate / "optimizers" / "adaptive_spectral_guard"
    if (nested / "adaptive_spectral_guard").is_dir():
        ROOT = nested
        break

if ROOT is None:
    raise RuntimeError("Open this notebook from a clone of rg_optimizers.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from adaptive_spectral_guard import (
    ControllerConfig,
    GuardConfig,
    MNISTGuardExperimentConfig,
    preset_policies,
    run_mnist_guard_comparison,
)
from adaptive_spectral_guard.plotting import (
    plot_controller,
    plot_corrections,
    plot_matched_convergence,
    plot_performance,
    plot_weightwatcher,
)
import adaptive_spectral_guard.experiment as experiment_module

print("Package root:", ROOT)
print("Torch:", torch.__version__)
print("WeightWatcher:", getattr(ww, "__version__", "unknown"))


## Stabilized V2 configuration

The confidence EMA uses a decay of $0.67$, which gives the current checkpoint approximately one-third of the weight and retains a short memory of the preceding two checkpoints.

When a controlled layer is active and $\alpha\leq2.05$, the volume confidence is floored at $0.25$. The shape channel has no such floor: it requires smoothed confidence of at least $0.15$ and raw confidence of at least $0.05$.


In [ ]:
PRESET = "stabilized"

experiment_config = MNISTGuardExperimentConfig(
    seed=1337,
    epochs=30,
    batch_size=128,
    learning_rate=1e-3,
    weight_decay=1e-2,
    grad_clip_norm=1.0,
    ww_min_evals=10,
    ww_max_evals=None,
    n_shells=5,
    min_beta_retained=20,
    min_beta_decades=0.50,
    train_eval_max_batches=None,
)

controller_config = ControllerConfig(
    alpha_on=2.08,
    alpha_strong=1.98,
    alpha_off=2.18,
    alpha_trend_on=-0.04,
    trend_ceiling=2.30,
    off_patience=2,

    min_confidence=0.20,
    support_change_scale=0.20,
    erg_gap_ratio_scale=0.30,

    confidence_ema_decay=0.67,
    separate_channel_confidence=True,
    volume_confidence_floor_below_boundary=0.25,
    volume_confidence_floor_alpha=2.05,
    shape_min_confidence=0.15,
    shape_raw_confidence_floor=0.05,

    beta_on=0.05,
    shape_alpha_on=2.05,
    shape_requires_alpha_boundary=True,

    task_conflict_ema_decay=0.80,
    task_conflict_penalty=2.0,
    minimum_task_throttle=0.10,
)

guard_config = GuardConfig(
    controller=controller_config,
    policies=preset_policies(PRESET),
)

RUN_STAMP = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = (
    ROOT
    / f"results_adaptive_spectral_guard_{PRESET}_30epochs_{RUN_STAMP}"
)

policy_table = pd.DataFrame(
    {
        name: vars(policy)
        for name, policy in guard_config.policies.items()
    }
).T
display(policy_table)
display(pd.Series(vars(controller_config), name="controller").to_frame())

print("Results:", OUTPUT_DIR.resolve())


## Expanded live epoch report

The base experiment already prints performance, WeightWatcher metrics, and correction summaries each epoch. This notebook adds a second live table showing the raw and smoothed ECS confidence, the distinct volume/shape confidence gates, and the resulting channel-specific gains.


In [ ]:
_base_epoch_report = experiment_module._print_epoch_report

def _stabilized_epoch_report(**kwargs):
    _base_epoch_report(**kwargs)
    controller_epoch = kwargs["controller_epoch"]
    if controller_epoch is None or controller_epoch.empty:
        return

    columns = [
        "parameter",
        "regime",
        "reason",
        "raw_confidence",
        "smoothed_confidence",
        "volume_confidence",
        "shape_confidence",
        "task_throttle",
        "volume_effective_gain",
        "shape_effective_gain",
        "shape_active",
        "policy_volume_max_ratio",
        "policy_shape_max_ratio",
        "policy_combined_max_ratio",
    ]
    available = [column for column in columns if column in controller_epoch.columns]
    print("", flush=True)
    print("STABILIZED V2 CHANNEL GATES FOR NEXT EPOCH", flush=True)
    print(
        controller_epoch[available].to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}",
        ),
        flush=True,
    )

experiment_module._print_epoch_report = _stabilized_epoch_report
print("Expanded V2 live reporting installed.")


## Run the 30-epoch paired experiment

The run can be stopped cleanly by creating the printed `STOP_AFTER_CURRENT_EPOCH` file from another terminal. A `KeyboardInterrupt` preserves all completed epochs.


In [ ]:
result = run_mnist_guard_comparison(
    experiment_config,
    guard_config,
    data_dir=ROOT / "data",
    output_dir=OUTPUT_DIR,
    progress=True,
)

print("Completed through epoch:", int(result.performance["epoch"].max()))
print("Saved to:", result.output_dir.resolve())


## Performance and WeightWatcher trajectories

In [ ]:
plot_performance(result.performance)

plot_weightwatcher(
    result.weightwatcher,
    "alpha",
    reference=2.0,
    title=r"All layers: WeightWatcher $\alpha$",
    ylabel=r"WeightWatcher $\alpha$",
)

plot_weightwatcher(
    result.weightwatcher,
    "ERG_gap",
    reference=0.0,
    title="All layers: WeightWatcher ERG gap",
    ylabel="WeightWatcher ERG gap",
)

plot_weightwatcher(
    result.weightwatcher,
    "beta_E_midpoint",
    reference=0.0,
    title=r"All layers: shell $\beta_E$",
    ylabel=r"Shell $\beta_E$",
)


## Raw versus smoothed confidence and channel-specific gains

These plots test whether V2 removes the epoch-to-epoch `strong → off → strong` switching seen in FC1. The volume gain should remain nonzero below the alpha boundary even when the raw shape confidence temporarily collapses.


In [ ]:
controller = result.controller.copy()
controller = controller.loc[controller["epoch"].ge(1)].copy()
controller["layer"] = (
    controller["parameter"]
    .astype(str)
    .str.replace(".weight", "", regex=False)
    .str.split(".")
    .str[-1]
)

for layer in ["fc1", "fc2"]:
    group = controller.loc[controller["layer"].eq(layer)].sort_values("epoch")
    if group.empty:
        continue

    fig, ax = plt.subplots(figsize=(10, 5), dpi=135)
    for metric, label in [
        ("raw_confidence", "raw confidence"),
        ("smoothed_confidence", "smoothed confidence"),
        ("volume_confidence", "volume confidence"),
        ("shape_confidence", "shape confidence"),
    ]:
        if metric in group:
            ax.plot(group["epoch"], group[metric], marker="o", linewidth=2, label=label)
    ax.axhline(
        controller_config.shape_min_confidence,
        linestyle="--",
        linewidth=1.2,
        label="shape minimum",
    )
    ax.set(
        xlabel="Epoch receiving the controller state",
        ylabel="Confidence",
        title=f"{layer.upper()}: raw, smoothed, and channel confidence",
    )
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5), dpi=135)
    ax.plot(
        group["epoch"],
        group["volume_effective_gain"],
        marker="o",
        linewidth=2.3,
        label="trace-log volume gain",
    )
    ax.plot(
        group["epoch"],
        group["shape_effective_gain"],
        marker="s",
        linewidth=2.3,
        label=r"shell-$\beta_E$ shape gain",
    )
    ax.set(
        xlabel="Epoch receiving the controller state",
        ylabel="Effective gain",
        title=f"{layer.upper()}: channel-specific effective gains",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

display(
    controller[
        [
            "epoch",
            "layer",
            "regime",
            "alpha",
            "raw_confidence",
            "smoothed_confidence",
            "volume_confidence",
            "shape_confidence",
            "volume_effective_gain",
            "shape_effective_gain",
            "shape_active",
        ]
    ].tail(20)
)


## Beta-channel saturation test

The first adaptive run had a shape-cap fraction near one whenever the shape channel was active. V2 should reduce both the mean correction ratio and the fraction of corrected steps that hit the shape cap.


In [ ]:
steps = result.guard_steps.copy()
if steps.empty:
    print("No guard-step diagnostics were recorded.")
    saturation = pd.DataFrame()
else:
    steps["layer"] = (
        steps["parameter"]
        .astype(str)
        .str.replace(".weight", "", regex=False)
        .str.split(".")
        .str[-1]
    )
    for column in [
        "shape_correction_ratio",
        "volume_correction_ratio",
        "combined_correction_ratio",
        "beta_E_local",
        "beta_E_excess",
    ]:
        if column in steps:
            steps[column] = pd.to_numeric(steps[column], errors="coerce")

    saturation = (
        steps.groupby(["epoch", "layer"], as_index=False)
        .agg(
            due_checks=("global_step", "size"),
            corrections_applied=("status", lambda x: x.eq("ok").sum()),
            mean_shape_ratio=("shape_correction_ratio", "mean"),
            max_shape_ratio=("shape_correction_ratio", "max"),
            shape_cap_fraction=("shape_capped", lambda x: x.fillna(False).astype(bool).mean()),
            mean_volume_ratio=("volume_correction_ratio", "mean"),
            volume_cap_fraction=("volume_capped", lambda x: x.fillna(False).astype(bool).mean()),
            mean_combined_ratio=("combined_correction_ratio", "mean"),
            combined_cap_fraction=("combined_capped", lambda x: x.fillna(False).astype(bool).mean()),
            mean_beta_E_local=("beta_E_local", "mean"),
            mean_beta_E_excess=("beta_E_excess", "mean"),
        )
    )
    display(saturation)

    for metric, title, ylabel in [
        (
            "mean_shape_ratio",
            "Mean shell-beta correction / AdamW step",
            "Mean shape-correction ratio",
        ),
        (
            "shape_cap_fraction",
            "Fraction of due checks hitting the shell-beta cap",
            "Shape-cap fraction",
        ),
    ]:
        fig, ax = plt.subplots(figsize=(10, 5), dpi=135)
        for layer in ["fc1", "fc2"]:
            group = saturation.loc[saturation["layer"].eq(layer)].sort_values("epoch")
            if not group.empty:
                ax.plot(group["epoch"], group[metric], marker="o", linewidth=2.3, label=layer.upper())
        ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
        if "fraction" in metric:
            ax.set_ylim(-0.02, 1.02)
        ax.grid(alpha=0.25)
        ax.legend()
        plt.tight_layout()
        plt.show()


## Controller, correction, and matched-convergence diagnostics

In [ ]:
display(result.controller.tail(16))
display(result.correction_summary.tail(16))

plot_controller(result.controller)
plot_corrections(result.correction_summary)
plot_matched_convergence(
    result.performance,
    result.weightwatcher,
)


## Direct-source and safety checks

The assertions below verify that alpha and ERG gap came directly from WeightWatcher and that the task-loss safeguard did not leave a positive first-order task conflict.


In [ ]:
ok = result.weightwatcher.loc[
    result.weightwatcher["status"].eq("ok")
].copy()

assert ok["alpha_source"].astype(str).eq("WeightWatcher").all()
assert ok["ERG_gap_source"].astype(str).eq("WeightWatcher").all()

if not result.guard_steps.empty:
    post_conflict = pd.to_numeric(
        result.guard_steps["task_conflict_ratio_post"],
        errors="coerce",
    ).dropna()
    print("Maximum post-safeguard task conflict:", post_conflict.max())
    print("Fraction positive after safeguard:", (post_conflict > 1e-6).mean())

below = controller.loc[
    controller["alpha"].lt(2.0)
    & controller["regime"].ne("off")
]
if not below.empty:
    assert below["volume_effective_gain"].gt(0.0).all()
    print(
        "All active alpha<2 controller states retained nonzero "
        "trace-log volume protection."
    )

display(
    ok[
        [
            "run",
            "epoch",
            "layer_name",
            "alpha",
            "ERG_gap",
            "beta_E_midpoint",
            "scale_balance_reliable",
        ]
    ].tail(18)
)


## Optional stabilized layer ablations

Set `RUN_ABLATIONS=True` to compare the stabilized full policy with stabilized FC1-only and FC2-only policies. This remains the cleanest way to distinguish direct FC2 overconstraint from an indirect FC2 effect caused by changing FC1.


In [ ]:
RUN_ABLATIONS = False
ABLATION_PRESETS = [
    "stabilized_fc1_only",
    "stabilized_fc2_only",
    "stabilized",
]
ablation_results = {}

if RUN_ABLATIONS:
    for preset in ABLATION_PRESETS:
        stamp = time.strftime("%Y%m%d_%H%M%S")
        ablation_output = (
            ROOT
            / f"results_adaptive_spectral_guard_ablation_{preset}_{stamp}"
        )
        print("\nRUNNING ABLATION:", preset)
        ablation_results[preset] = run_mnist_guard_comparison(
            experiment_config,
            GuardConfig(
                controller=controller_config,
                policies=preset_policies(preset),
            ),
            data_dir=ROOT / "data",
            output_dir=ablation_output,
            progress=True,
        )

    rows = []
    for preset, ablation in ablation_results.items():
        final = (
            ablation.performance
            .sort_values("epoch")
            .groupby("run")
            .tail(1)
        )
        for _, row in final.iterrows():
            rows.append(
                {
                    "preset": preset,
                    "run": row["run"],
                    "epoch": row["epoch"],
                    "train_loss": row["train_loss"],
                    "train_acc": row["train_acc"],
                    "test_loss": row["test_loss"],
                    "test_acc": row["test_acc"],
                }
            )
    display(pd.DataFrame(rows))


## Interpretation

V2 succeeds at the controller level when:

- FC1 does not alternate between `strong` and `off` merely because one raw-confidence checkpoint is unstable;
- the trace-log volume gain remains nonzero below the boundary;
- the shape channel turns off on severely unstable checkpoints;
- the shape-cap fraction is substantially below one, or at least the applied cap is materially smaller than in V1;
- FC2 retains convergence while the guarded model preserves or improves test performance at matched training progress.

It is still an empirical optimizer experiment. A better spectral trajectory is not sufficient by itself; the decisive evidence is lower test loss or better test accuracy at matched train loss/accuracy.
